<a href="https://colab.research.google.com/github/czhang0673/vit-art-classifier/blob/main/vit_art_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The core architecture of this model is built using **PyTorch** and **Torchvision**. **Scikit-learn** and **Matplotlib** are utilized for post-training evaluation (confusion matrices and loss curves). Standard Python utility modules (`os`, `shutil`, `random`) are used to dynamically construct the training and validation splits from the raw image directory.

In [ ]:
#Configuration and File Paths:
#NOTE: The original 1,200-image dataset is not included in this repository due to size constraints.
#To reproduce this training pipeline, place your image data in a folder named 'artdata' in the same directory as this notebook.

DATA_DIR = "./artdata"
SPLIT_DIR = "./artdata_split"
TEST_DIR = "./test_images" #The 6 sample images included in the repo
SAVE_DIR = "./models"

MODEL_WEIGHTS_PATH = ".model_weights.pth"

#Check if the data directory exists
if not os.path.exists(DATA_DIR):
    print(f"Warning: The directory '{DATA_DIR}' was not found.")
    print("Inference can still be run on the test images, but training blocks will fail.")
else:
    print(f"Training dataset located at {DATA_DIR}. Ready to proceed.")

NameError: name 'os' is not defined

In [ ]:
#Setup and Initalization: This section imports the necessary libraries for data processing, model training, and evaluation.
#Standard Library Imports
import random
import os
from datetime import datetime
import shutil
import warnings

#Data Manipulation and Visualization
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from PIL import Image

#Core PyTorch and Neural Network Modules
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

#Computer Vision and Image Handling
import torchvision
from torchvision.datasets import ImageFolder
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, Subset
 #Pil, nn.functional for single image input function

#Suppress Pillow warning regarding "P mode" transparency.
#PyTorch transforms automatically convert these to standard RGB/RGBA tensors.
warnings.filterwarnings(
    "ignore",
    message="Palette images with Transparency expressed in bytes should be converted to RGBA images"
)

# Verify PyTorch environment
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")

In [ ]:
#Creates separate directory to hold training images, validation images, copies them over from original artdata
random.seed(42)

#Checks if source data exists
if not os.path.exists(DATA_DIR):
    print(f"Skipping split: Source directory '{DATA_DIR}' not found.")

#Prevent duplicates if split already exists
elif os.path.exists(SPLIT_DIR):
    print(f"Directory '{SPLIT_DIR}' already exists. Skipping data partitioning to prevent duplicates.")
else:
  print(f"Partitioning data from '{DATA_DIR}' to '{SPLIT_DIR}'")

  os.makedirs(SPLIT_DIR, exist_ok=True)

  train_dir = os.path.join(SPLIT_DIR, "train")
  val_dir = os.path.join(SPLIT_DIR, "val")

  genres = [d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))]

  print("Starting the data split...")

  for genre in genres:
      genre_path = os.path.join(DATA_DIR, genre)
      images = [f for f in os.listdir(genre_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

      random.shuffle(images)

      #90/10 split
      split_index = int(len(images) * 0.9)
      train_images = images[:split_index]
      val_images = images[split_index:]

      genre_train_dir = os.path.join(train_dir, genre)
      genre_val_dir = os.path.join(val_dir, genre)

      os.makedirs(genre_train_dir, exist_ok=True)
      os.makedirs(genre_val_dir, exist_ok=True)

      for img in train_images:
          shutil.copy(os.path.join(genre_path, img), os.path.join(genre_train_dir, img))

      for img in val_images:
          shutil.copy(os.path.join(genre_path, img), os.path.join(genre_val_dir, img))

      print(f"{genre}: Copied {len(train_images)} to train, {len(val_images)} to val.")

In [ ]:
#Training transforms introduce spatial and color variance to prevent overfitting.
#ImageNet normalization applied to match the pre-trained ViT weight distribution.
train_transforms = transforms.Compose([
  transforms.RandomResizedCrop(224),
  transforms.RandomRotation(20),
  transforms.RandomHorizontalFlip(),
  transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
  transforms.ToTensor(),
  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#Validation transforms: Strict resizing and center cropping for standardized evaluation.ImageNet normalization applied.
val_transforms = transforms.Compose([
  transforms.Resize(256),
  transforms.CenterCrop(224),
  transforms.ToTensor(),
  transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

#Define paths relative to the partitioned data directory. Ensure SPLIT_DIR was created in the previous block.
train_path = os.path.join(SPLIT_DIR, "train")
val_path = os.path.join(SPLIT_DIR, "val")

#Initialize datasets
train_dataset = datasets.ImageFolder(root=train_path, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=val_path, transform=val_transforms)

class_names = train_dataset.classes

#Initalize DataLoaders. Batch size is 64
#pin_memory = True facilitates data transfer to GPU
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers = 2, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers = 2, pin_memory = True)

print(f"Total training images: {len(train_dataset)}")
print(f"Total validation images: {len(val_dataset)}")
print(f"Detected Classes: {class_names}")


NameError: name 'transforms' is not defined

Data Validation & Class Distribution:

Before initializing the model, verify the integrity of the DataLoaders and ensure the dataset is equally balanced across all 6 art genres. This prevents the model from developing a class bias during training.

In [ ]:
#Confirmation of readable images per genre:
print("CLASS DISTRIBUTION: TRAINING SET")
print("-" * 40)
for i, class_name in enumerate(train_dataset.classes):
    count = train_dataset.targets.count(i)
    print(f"{class_name.ljust(15)}: {count} images")
print("-" * 40)
print(f"Total Training Images: {len(train_dataset)}\n")


print("CLASS DISTRIBUTION: VALIDATION SET")
print("-" * 40)
for i, class_name in enumerate(val_dataset.classes):
    count = val_dataset.targets.count(i)
    print(f"{class_name.ljust(15)}: {count} images")
print("-" * 40)
print(f"Total Validation Images: {len(val_dataset)}")

In [ ]:
#Training and Validation Functions:
def train(model, loader, criterion, optimizer, device):
    """
    Executes one epoch of training over the entire dataloader.

    Args:
        model: The PyTorch model being trained.
        loader: DataLoader containing the training dataset.
        criterion: The loss function (ex. CrossEntropyLoss).
        optimizer: The optimization algorithm.
        device: The compute device ('cuda' or 'cpu').

    Returns:
        Tuple containing the average epoch loss and epoch accuracy.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        #Calculating batch accuracy
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0) #Dynamically scales, which accounts for smaller final batch sizes
        correct += (predicted == labels).sum().item()
        running_loss += loss.item()

    epoch_loss = running_loss / len(loader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy

def validate(model, loader, criterion, device):
    """
    Evaluates the model on the validation dataset without tracking gradients.

    Returns:
        Tuple containing the average validation loss and validation accuracy.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            running_loss += loss.item()

    epoch_loss = running_loss / len(loader)
    accuracy = 100 * correct / total

    return epoch_loss, accuracy

In [ ]:
#Model Initialization:
#Load pre-trained Vision Transformer weights
weights = models.ViT_B_16_Weights.DEFAULT
model = models.vit_b_16(weights = weights)

#Replace classification head for 6 classes. ViT backbone left unfrozen for end-to-end fine-tuning
#Vit-B/16 divides 224 x 224 image into 14 x 14 patches (196 total)
model.heads.head = nn.Linear(model.heads.head.in_features, len(train_dataset.classes))

#Device Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

#Hyperparameters and Tracking:
num_epochs = 50
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1E-5, weight_decay = 0.01)

#Learning Rate Scheduler: Reduces LR by a factor of 10 if validation loss stagnates for 3 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=3
)

#Early Stopping and Checkpointing
patience = 5
unimproved_epochs = 0
best_val_loss = float('inf')

#Save path configuration
os.makedirs(SAVE_DIR, exist_ok = True)
current_date = datetime.now().strftime("%m-%d-%y")

print(f"Model initialized on device: {device}")

In [ ]:
#Training Loop with Early Stopping:

print("Starting training loop...")
#track the past of the last saved model. Prevents local storage bloat
last_saved_path = None
num_classes = len(class_names)

#History dictionary for performance visualization
history = {
    'epoch': [],
    'train_loss': [],
    'val_loss': [],
    'train_acc': [],
    'val_acc': []
}

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    print(f"Epoch [{epoch+1}/{num_epochs}], "
          f"Train Loss: {train_loss:.4f}, Train Accuracy: {train_acc:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc:.2f}%")

    #Update history dictionaries for plotting
    history['epoch'].append(epoch + 1)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    #Step learning rate scheduler
    scheduler.step(val_loss)

    #Save model state if validation loss reaches a new minimum
    if val_loss < best_val_loss:
        print(f"Validation loss decreased ({best_val_loss:.4f} --> {val_loss:.4f}). Saving model...")
        best_val_loss = val_loss
        unimproved_epochs = 0

        #Save new model weights
        filename = f"vit_art_classification_{num_classes}_{val_acc:.2f}acc_{current_date}.pth"
        save_path = os.path.join(SAVE_DIR, filename)
        torch.save(model.state_dict(), save_path)

        #Delete previously saved model to conserve space
        if last_saved_path and os.path.exists(last_saved_path):
          os.remove(last_saved_path)
        last_saved_path = save_path

    else:
        unimproved_epochs += 1
        print(f"Epoch {epoch + 1}: No improvement. {unimproved_epochs} of {patience}.")
        if unimproved_epochs >= patience:
            print(f"Validation loss hasn't improved in {patience} epochs. Early stopping triggered.")
            break

Evaluation and Inference

Load optimal weights to generate performance metrics and test individual images of model at its lowest validation loss.

In [ ]:
#Load Optimal Weights:

#Determine correct path. If the training loop was just run, use newly generated model
#Otherwise, use pre-trained weights specified in config block
if 'last_saved_path' in locals() and last_saved_path and os.path.exists(last_saved_path):
  load_path = last_saved_path
else:
  load_path = MODEL_WEIGHTS_PATH
try:
  print(f"Loading weights from: {load_path}")

  #map_location = device safely handles transition if moving from a GPU training environment to a CPU evaluation environment
  model.load_state_dict(torch.load(load_path, map_location = device))

  print("Optimal model weighs successfully loaded. Ready for evaluation.")
except FileNotFoundError:
  print(f"Error: Could not find model weights at '{load_path}'.")
  print("Please ensure you have either run the training loop or downloaded the pre-trained .pth file as instructed in the README.")

Training complete. Loading the best model weights...


NameError: name 'model' is not defined

In [ ]:
def plot_training_history(history_dict):
    """
    Converts the training history dictionary into a pandas DataFrame
    and plots the Loss and Accuracy learning curves.
    """
    df = pd.DataFrame(history_dict)

    # Initialize 1 x 2 figure with loss on left, accuracy on right
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot loss
    axes[0].plot(df['epoch'], df['train_loss'], label='Train Loss', color='blue', linewidth=2)
    axes[0].plot(df['epoch'], df['val_loss'], label='Validation Loss', color='orange', linestyle='--', linewidth=2)
    axes[0].set_title('Model Loss Progress')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True, linestyle=':', alpha=0.6)

    # Plot accuracy
    axes[1].plot(df['epoch'], df['train_acc'], label='Train Accuracy', color='blue', linewidth=2)
    axes[1].plot(df['epoch'], df['val_acc'], label='Validation Accuracy', color='orange', linestyle='--', linewidth=2)
    axes[1].set_title('Model Accuracy Progress')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout() # Dynamically adjusts padding to prevent label overlap
    plt.show()

# Executes plotting
plot_training_history(history)

In [ ]:
def plot_confusion_matrix(model, loader, device, class_names):
  """
  Evaluates the model on the validation set and plots a confusion matrix to visualize true vs. predicted classifications across all art genres.
  """
  all_preds = []
  all_labels = []

  model.eval()
  with torch.no_grad():
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        output = model(images)
        _, predicted = torch.max(output, 1)

        #Transfer tensors to CPU memory for Sci-kit learn compatibility
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    #Generate and plot matrix
    cm = confusion_matrix(all_labels, all preds)
    displ = ConfusionMatrixDisplay(confusion_matrix = cm, display_labels = class_names)

    fig, ax = plt.subplots(figsize = (8,8))
    disp.plot(cmap = plt.cm.Blues, ax = ax, xticks_rotation = 45)

    plt.title("Validation Confusion Matrix")
    plt.tight_layout()
    plt.show()

#Execute function using dynamically loaded class names:
plot_confusion_matrix(model, val_loader, device, val_dataset.classes)

SyntaxError: invalid syntax. Perhaps you forgot a comma? (35345614.py, line 21)

In [ ]:
#Visual Error Analysis

def show_misclassified_images(dataset, all_preds, all_labels, class_names, num_images = 4):

"""
Identifies misclassified images from the validation set and displays them alongside their true and predicted labels. Dymanically scales plot grid; displays a maximum of 4 misclassified images.
"""
#Use list comprehension for faster, cleaner filtering:
misclassified_indicies = [i for i in range(len(all_preds)) if all_preds[i] != all_labels[i]]

print(f"Found {len(misclassified_indicies)} misclassified images.")

#Dynamically determine how many images to plot
display_count = min(num_images, len(misclassfied_indices))

if display_count == 0:
  print("Model accuracy is 100%. No misclassified images to display.")
  return

#Scale the width of the figure based on number of images(4 inches per image)
fig, axes = plt.subplots(1, display_count, figsize = (4 * display_count, 5))

#Safety catch for Matplotlib: If only 1 image is misclassified, wrap single axis in a list so it remains iterable
if display_count == 1:
  axes = [axes]

#Isolate denormalization variables so they do not rely on global scopes
mean = torch.tensor([0.485, 0.456, 0.406])
std = torch.tensor([0.229, 0.224, 0.225])

for i, idx in enumerate(misclassified_indices[:display_count]):
  img, label = dataset[idx]
  img = img.permute(1, 2, 0)

  #Reverse normalization for accurate visual representation
  img = img * std + mean
  img = torch.clamp(img, 0, 1)

  axes[i].imshow(img)
  axes[i].set_title(f"True: {class_names[all_labels[idx]]}\nPred: {class_names[all_preds[idx]]}", color = 'red')
  axes[i].axis('off')

plt.tight_layout()
plt.show()

#Execute using dynamic variables
show_misclassified_images(val_dataset, all_preds, all_labesl, val_dataset.classes)

In [ ]:
#Single Image Inference:
def predict_single_image(image_path, model, transform, device, class_names):
  """
  Loads a single image form a filepath, applies validation transforms, and returns the predicted class and confidence percentage.
  """
  img = Image.open(image_path).convert('RGB') #.convert('RGB') ensures 3 color channels. Prevents crashes on grayscale image

  img_tensor = transform(img)
  img_tensor = img_tensor.unsqueeze(0)
  img_tensor = img_tensor.to(device)

  model.eval()
  with torch.no_grad():
    raw_outputs = model(img_tensor)
    probabilities = F.softmax(raw_outputs, dim = 1)
    confidence, predicted_index = torch.max(probabilities, 1)

    confidence_pct = confidence.item() * 100
    predicted_class = class_names[predicted_index.item()]

    return predicted_class, confidence_pct


print("Running Single Image Testing:")

valid_extensions = ".jpg", ".jpeg", ".png"

try:
  all_files = [f for f in os.listdir(TEST_DIR) if f.lower().endswith(valid_extensions)]
except FileNotFoundError:
  all_files = []
  print(f"Error: The directory '{TEST_DIR}' does not exist. Please check your paths.")

if not all_files:
  print("Error: No image files found in the specified testing folder.")
else:
    random_filename = random.choice(all_files)
    random_image_path = os.path.join(TEST_DIR, random_filename)

    print(f"Testing {random_filename}")


    predicted_style, confidence = predict_single_image(image_path = random_image_path, model = model, transform = val_transforms, device = device, class_names = val_dataset.classes)

    #Display the result
    display_img = Image.open(random_image_path)

    fig, axes = plt.subplots(1, 1, figsize = (6, 6))
    axes.imshow(display_img)
    axes.axis('off')

    plt.tight_layout()
    plt.show()
    print(f"Predicted Class: {predicted_style} ({confidence:.2f}% Confidence)")
